In [ ]:
# Set up Step 4.
#
# Step 4 reads Step 3 scene-level JSON files from:
#   cfg.STEP4_INPUT_DIR / FloorPlanX_step3_text_{RUN_MODE}.json
#
# It writes flat pair-level probe samples and composition /
# implicit-transitive probe samples.

import os
import sys
import shutil
import importlib
from google.colab import files, drive

drive.mount("/content/drive")


PILOT_ROOT = "/content/pilot_full"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")


print("Upload config.py / config_positional_context_full_drive.py")
uploaded = files.upload()

config_names = [
    name for name in uploaded.keys()
    if name.startswith("config") and name.endswith(".py")
]

if len(config_names) != 1:
    raise ValueError(
        f"Expected exactly one config*.py file, got: {config_names}"
    )

config_src = os.path.join("/content", config_names[0])
config_dst = os.path.join(PILOT_ROOT, "config.py")
shutil.copy(config_src, config_dst)


if "pilot_full.config" in sys.modules:
    del sys.modules["pilot_full.config"]

import pilot_full.config as cfg
importlib.reload(cfg)

os.makedirs(cfg.STEP4_OUTPUT_DIR, exist_ok=True)

print("\nLoaded config:", config_names[0])
print("PILOT_ROOT:", cfg.PILOT_ROOT)
print("PIPELINE_DATA_DIR:", cfg.PIPELINE_DATA_DIR)
print("RUN_MODE:", cfg.RUN_MODE)
print("STEP4_INPUT_SOURCE:", cfg.STEP4_INPUT_SOURCE)
print("STEP4_INPUT_DIR:", cfg.STEP4_INPUT_DIR)
print("STEP4_SCENE_JSON_SUFFIX:", cfg.STEP4_SCENE_JSON_SUFFIX)

print("\nPair-level Step4 outputs:")
print("STEP4_OUTPUT_DIR:", cfg.STEP4_OUTPUT_DIR)
print("STEP4_OUTPUT_FILE:", cfg.STEP4_OUTPUT_FILE)
print("STEP4_MANIFEST_FILE:", cfg.STEP4_MANIFEST_FILE)
print("STEP4_PREVIEW_CSV_FILE:", cfg.STEP4_PREVIEW_CSV_FILE)

print("\nComposition Step4 outputs:")
print("STEP4_COMPOSITION_OUTPUT_FILE:", cfg.STEP4_COMPOSITION_OUTPUT_FILE)
print("STEP4_COMPOSITION_MANIFEST_FILE:", cfg.STEP4_COMPOSITION_MANIFEST_FILE)
print("STEP4_COMPOSITION_PREVIEW_CSV_FILE:", cfg.STEP4_COMPOSITION_PREVIEW_CSV_FILE)

Mounted at /content/drive
Upload config.py / config_positional_context_full_drive.py


Saving config_positional_context_full_drive.py to config_positional_context_full_drive.py

Loaded config: config_positional_context_full_drive.py
PILOT_ROOT: /content/pilot_full
PIPELINE_DATA_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data
RUN_MODE: diverse
STEP4_INPUT_SOURCE: scene_json
STEP4_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_div
STEP4_SCENE_JSON_SUFFIX: _step3_text_diverse.json

Pair-level Step4 outputs:
STEP4_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe
STEP4_OUTPUT_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_probe_samples_diverse_all.jsonl
STEP4_MANIFEST_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_manifest_diverse.json
STEP4_PREVIEW_CSV_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe

In [ ]:
import os
import json
import re
import random
import hashlib
import shutil
import importlib

from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

import pilot_full.config as cfg
importlib.reload(cfg)

from pilot_full.config import (
    RANDOM_SEED,
    SCENES,
    RUN_MODE,

    STEP4_INPUT_SOURCE,
    STEP4_INPUT_DIR,
    STEP4_SCENE_JSON_SUFFIX,
    STEP4_OUTPUT_DIR,
    STEP4_OUTPUT_FILE,
    STEP4_MANIFEST_FILE,
    STEP4_PREVIEW_CSV_FILE,
    STEP4_KEEP_EVIDENCE_TYPES,
    STEP4_INCLUDE_NONE_LABEL,
    STEP4_WRITE_PER_SCENE_FILES,

    STEP4_COMPOSITION_OUTPUT_FILE,
    STEP4_COMPOSITION_MANIFEST_FILE,
    STEP4_COMPOSITION_PREVIEW_CSV_FILE,
    STEP4_WRITE_COMPOSITION_PER_SCENE_FILES,
    STEP4_COMPOSITION_TARGET_RELATIONS,
    STEP4_COMPOSITION_REQUIRE_TARGET_IMPLICIT_LABELED,
    STEP4_COMPOSITION_EXCLUDE_DIRECT_OR_INVERSE_EXPLICIT_TARGET,
    STEP4_COMPOSITION_EXCLUDE_ANY_EXPLICIT_TARGET_PAIR,
    STEP4_COMPOSITION_REQUIRE_UNIQUE_SUPPORT_PATH,
    STEP4_COMPOSITION_KEEP_ALL_SUPPORT_PATHS,
    STEP4_COMPOSITION_ALLOWED_RULES,

    NONE_RELATION_LABEL,
    INVERSE_RELATION_MAP,

    EXPLICIT_DIRECT,
    EXPLICIT_INVERSE,
    IMPLICIT_LABELED,
    IMPLICIT_NONE,
    IMPLICIT_TRANSITIVE,

    MAIN_PROBE_EVIDENCE_TYPES,
)

random.seed(RANDOM_SEED)
os.makedirs(STEP4_OUTPUT_DIR, exist_ok=True)

print("Step 4 config loaded.")
print("RUN_MODE:", RUN_MODE)
print("STEP4_INPUT_SOURCE:", STEP4_INPUT_SOURCE)
print("STEP4_INPUT_DIR:", STEP4_INPUT_DIR)
print("STEP4_SCENE_JSON_SUFFIX:", STEP4_SCENE_JSON_SUFFIX)

print("\nPair-level output:")
print("STEP4_OUTPUT_FILE:", STEP4_OUTPUT_FILE)
print("STEP4_KEEP_EVIDENCE_TYPES:", STEP4_KEEP_EVIDENCE_TYPES)
print("STEP4_INCLUDE_NONE_LABEL:", STEP4_INCLUDE_NONE_LABEL)

print("\nComposition output:")
print("STEP4_COMPOSITION_OUTPUT_FILE:", STEP4_COMPOSITION_OUTPUT_FILE)
print("STEP4_COMPOSITION_TARGET_RELATIONS:", STEP4_COMPOSITION_TARGET_RELATIONS)
print("STEP4_COMPOSITION_ALLOWED_RULES:", STEP4_COMPOSITION_ALLOWED_RULES)
print("STEP4_COMPOSITION_REQUIRE_UNIQUE_SUPPORT_PATH:", STEP4_COMPOSITION_REQUIRE_UNIQUE_SUPPORT_PATH)

Step 4 config loaded.
RUN_MODE: diverse
STEP4_INPUT_SOURCE: scene_json
STEP4_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_div
STEP4_SCENE_JSON_SUFFIX: _step3_text_diverse.json

Pair-level output:
STEP4_OUTPUT_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_probe_samples_diverse_all.jsonl
STEP4_KEEP_EVIDENCE_TYPES: {'explicit_inverse_or_same_sentence_pair', 'implicit_labeled', 'implicit_none', 'explicit_direct'}
STEP4_INCLUDE_NONE_LABEL: True

Composition output:
STEP4_COMPOSITION_OUTPUT_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_composition_samples_diverse_all.jsonl
STEP4_COMPOSITION_TARGET_RELATIONS: {'below', 'right_of', 'above', 'left_of'}
STEP4_COMPOSITION_ALLOWED_RULES: {'chain_same_direction', 'anchor_between_reversed_surface_form', 'shared_anchor_opposite_sides'}
STEP4_COMPOSITION_REQUIRE_UNIQUE_SUPPORT_

In [ ]:
# Utility functions

def natural_sort_key(text: str):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", text)
    ]


def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: str, data: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def write_jsonl(path: str, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def stable_hash_text(text: str, n: int = 16) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:n]


def inverse_relation(label: str) -> str:
    return INVERSE_RELATION_MAP.get(label, label)


def unordered_pair_key(uid_a: str, uid_b: str) -> str:
    a, b = sorted([uid_a, uid_b])
    return f"{a}|||{b}"


def make_unordered_pair(uid_a: str, uid_b: str) -> frozenset:
    return frozenset([uid_a, uid_b])


def make_relation_key(subject_uid: str, relation: str, object_uid: str) -> Tuple[str, str, str]:
    return (subject_uid, relation, object_uid)


def find_all_spans(text: str, alias: str) -> List[Tuple[int, int]]:
    """
    Find all exact alias occurrences in text.

    Alias pattern avoids matching substrings inside longer aliases.
    Example:
        mug_1 should not match big_mug_1 or mug_10.
    """
    pattern = rf"(?<![A-Za-z0-9_]){re.escape(alias)}(?![A-Za-z0-9_])"

    return [
        (m.start(), m.end())
        for m in re.finditer(pattern, text)
    ]


print("Utility functions loaded.")

Utility functions loaded.


In [ ]:
# Pair-group and sample-ID helpers.

def make_pair_group_id(
    paragraph_id: str,
    subject_uid: str,
    object_uid: str,
    relation_label: str,
    pair_evidence_type: str,
    source_relation_summary: Optional[Dict[str, Any]],
) -> str:
    """
    Build a stable pair_group_id.

    For explicit direct/inverse pairs:
        use source_relation_summary so direct and inverse samples from the
        same explicit text relation share one group id.

    For implicit_labeled / implicit_none:
        use unordered object pair within the paragraph.
    """
    if pair_evidence_type in {EXPLICIT_DIRECT, EXPLICIT_INVERSE} and source_relation_summary:
        src_subject = source_relation_summary.get("subject_id")
        src_relation = source_relation_summary.get("relation")
        src_object = source_relation_summary.get("object_id")

        base = (
            f"{paragraph_id}|explicit|"
            f"{src_subject}|{src_relation}|{src_object}"
        )

        return "pg_" + stable_hash_text(base, n=20)

    base = (
        f"{paragraph_id}|implicit|"
        f"{unordered_pair_key(subject_uid, object_uid)}|{relation_label}"
    )

    return "pg_" + stable_hash_text(base, n=20)


def infer_probe_direction_relative_to_text(pair_evidence_type: str) -> str:
    if pair_evidence_type == EXPLICIT_DIRECT:
        return "direct"

    if pair_evidence_type == EXPLICIT_INVERSE:
        return "inverse"

    if pair_evidence_type == IMPLICIT_LABELED:
        return "implicit_labeled"

    if pair_evidence_type == IMPLICIT_NONE:
        return "implicit_none"

    if pair_evidence_type == IMPLICIT_TRANSITIVE:
        return "implicit_transitive"

    return "unknown"


def make_sample_id(
    paragraph_id: str,
    pair_index: int,
    subject_uid: str,
    relation_label: str,
    object_uid: str,
    pair_evidence_type: str,
) -> str:
    base = (
        f"{paragraph_id}|{pair_index}|"
        f"{subject_uid}|{relation_label}|{object_uid}|{pair_evidence_type}"
    )

    return "s4_" + stable_hash_text(base, n=24)


def make_composition_sample_id(
    paragraph_id: str,
    target_subject_uid: str,
    target_relation: str,
    target_object_uid: str,
) -> str:
    base = (
        f"{paragraph_id}|composition|"
        f"{target_subject_uid}|{target_relation}|{target_object_uid}"
    )

    return "s4t_" + stable_hash_text(base, n=24)


def make_composition_group_id(
    paragraph_id: str,
    target_subject_uid: str,
    target_relation: str,
    target_object_uid: str,
) -> str:
    base = (
        f"{paragraph_id}|composition_group|"
        f"{target_subject_uid}|{target_relation}|{target_object_uid}"
    )

    return "cpg_" + stable_hash_text(base, n=24)


print("Pair group helpers loaded.")

Pair group helpers loaded.


In [ ]:
# Load Step 3 scene-level JSON files one by one.
#
# Expected file pattern: FloorPlanX_step3_text_{RUN_MODE}.json
# The combined step3_paragraphs_{RUN_MODE}_all.jsonl file is not used here.

if STEP4_INPUT_SOURCE != "scene_json":
    raise ValueError(
        f"This notebook is configured to read Step3 scene JSON files only. "
        f"Expected STEP4_INPUT_SOURCE='scene_json', got: {STEP4_INPUT_SOURCE}"
    )

if not os.path.exists(STEP4_INPUT_DIR):
    raise FileNotFoundError(
        f"Step4 input directory does not exist:\n{STEP4_INPUT_DIR}"
    )

step3_scene_files = sorted(
    [
        filename
        for filename in os.listdir(STEP4_INPUT_DIR)
        if filename.startswith("FloorPlan")
        and filename.endswith(STEP4_SCENE_JSON_SUFFIX)
    ],
    key=natural_sort_key,
)

if not step3_scene_files:
    raise FileNotFoundError(
        f"No Step3 scene JSON files found in:\n{STEP4_INPUT_DIR}\n\n"
        f"Expected files ending with:\n{STEP4_SCENE_JSON_SUFFIX}"
    )

available_scenes = [
    filename.replace(STEP4_SCENE_JSON_SUFFIX, "")
    for filename in step3_scene_files
]

expected_scene_set = set(SCENES)
available_scene_set = set(available_scenes)

missing_scenes = sorted(
    expected_scene_set - available_scene_set,
    key=natural_sort_key,
)

extra_scenes = sorted(
    available_scene_set - expected_scene_set,
    key=natural_sort_key,
)

print("Step3 scene JSON files found:", len(step3_scene_files))
print("Available scenes:", len(available_scenes))
print("Configured scenes:", len(SCENES))
print("Missing configured scenes:", len(missing_scenes))
print("Extra scenes:", len(extra_scenes))

if missing_scenes:
    print("\n[Info] Configured scenes missing Step3 JSON files:")
    for scene in missing_scenes:
        print("-", scene)

if extra_scenes:
    print("\n[Warning] Step3 JSON files not listed in cfg.SCENES:")
    for scene in extra_scenes:
        print("-", scene)

print("\nFirst files:", step3_scene_files[:5])
print("Last files:", step3_scene_files[-5:])

Step3 scene JSON files found: 118
Available scenes: 118
Configured scenes: 120
Missing configured scenes: 2
Extra scenes: 0

[Info] Configured scenes missing Step3 JSON files:
- FloorPlan312
- FloorPlan315

First files: ['FloorPlan1_step3_text_diverse.json', 'FloorPlan2_step3_text_diverse.json', 'FloorPlan3_step3_text_diverse.json', 'FloorPlan4_step3_text_diverse.json', 'FloorPlan5_step3_text_diverse.json']
Last files: ['FloorPlan426_step3_text_diverse.json', 'FloorPlan427_step3_text_diverse.json', 'FloorPlan428_step3_text_diverse.json', 'FloorPlan429_step3_text_diverse.json', 'FloorPlan430_step3_text_diverse.json']


In [ ]:
def validate_step3_scene_schema(scene_data: Dict[str, Any], path: str) -> None:
    required_scene_keys = {
        "scene",
        "scene_type",
        "generation_mode",
        "num_paragraphs",
        "paragraphs",
    }

    missing = [
        key for key in required_scene_keys
        if key not in scene_data
    ]

    if missing:
        raise KeyError(
            f"Step3 scene JSON missing keys {missing}:\n{path}"
        )

    if not isinstance(scene_data["paragraphs"], list):
        raise TypeError(
            f"scene_data['paragraphs'] must be a list:\n{path}"
        )


def validate_step3_paragraph_schema(paragraph: Dict[str, Any]) -> None:
    required_paragraph_keys = {
        "paragraph_id",
        "scene",
        "scene_type",
        "chunk_id",
        "cluster_id",
        "generation_mode",
        "text",
        "object_mentions",
        "explicit_relations_in_text",
        "pair_targets",
    }

    missing = [
        key for key in required_paragraph_keys
        if key not in paragraph
    ]

    if missing:
        raise KeyError(
            f"Step3 paragraph missing keys {missing}; "
            f"paragraph_id={paragraph.get('paragraph_id')}"
        )

    if not isinstance(paragraph["explicit_relations_in_text"], list):
        raise TypeError(
            f"paragraph['explicit_relations_in_text'] must be a list; "
            f"paragraph_id={paragraph.get('paragraph_id')}"
        )

    if not isinstance(paragraph["pair_targets"], list):
        raise TypeError(
            f"paragraph['pair_targets'] must be a list; "
            f"paragraph_id={paragraph.get('paragraph_id')}"
        )


print("Step3 schema validation functions loaded.")

Step3 schema validation functions loaded.


In [ ]:
# Build one Step 4 pair-level probe sample from a Step 3 pair target.

def build_step4_sample(
    paragraph_record: Dict[str, Any],
    pair_target: Dict[str, Any],
    pair_index: int,
) -> Dict[str, Any]:
    paragraph_id = paragraph_record["paragraph_id"]
    text = paragraph_record["text"]

    subject_uid = pair_target["subject_uid"]
    object_uid = pair_target["object_uid"]

    relation_label = pair_target["relation_label"]
    pair_evidence_type = pair_target["pair_evidence_type"]

    source_relation_summary = pair_target.get("source_relation_summary")

    pair_group_id = make_pair_group_id(
        paragraph_id=paragraph_id,
        subject_uid=subject_uid,
        object_uid=object_uid,
        relation_label=relation_label,
        pair_evidence_type=pair_evidence_type,
        source_relation_summary=source_relation_summary,
    )

    sample_id = make_sample_id(
        paragraph_id=paragraph_id,
        pair_index=pair_index,
        subject_uid=subject_uid,
        relation_label=relation_label,
        object_uid=object_uid,
        pair_evidence_type=pair_evidence_type,
    )

    probe_direction = infer_probe_direction_relative_to_text(
        pair_evidence_type
    )

    subject_all_char_spans = find_all_spans(text, subject_uid)
    object_all_char_spans = find_all_spans(text, object_uid)

    has_relation_label = pair_target.get(
        "has_relation_label",
        relation_label != NONE_RELATION_LABEL,
    )

    is_inverse_of_text_relation = pair_evidence_type == EXPLICIT_INVERSE

    row = {
        "sample_id": sample_id,
        "sample_type": "pair",
        "pair_group_id": pair_group_id,

        "paragraph_id": paragraph_id,
        "scene": paragraph_record["scene"],
        "scene_type": paragraph_record.get("scene_type"),
        "chunk_id": paragraph_record["chunk_id"],
        "cluster_id": paragraph_record["cluster_id"],
        "generation_mode": paragraph_record.get("generation_mode"),
        "paragraph_index_within_cluster": paragraph_record.get(
            "paragraph_index_within_cluster"
        ),

        # Text input for Step5
        "text": text,

        # Subject / object pair
        "subject_uid": subject_uid,
        "subject_id": pair_target.get("subject_id"),
        "subject_type": pair_target.get("subject_type"),

        "object_uid": object_uid,
        "object_id": pair_target.get("object_id"),
        "object_type": pair_target.get("object_type"),

        # Label fields for Step6
        "relation": relation_label,
        "relation_label": relation_label,
        "has_relation_label": has_relation_label,
        "label_is_none": relation_label == NONE_RELATION_LABEL,

        # Evidence fields
        "relation_source": pair_target.get("relation_source"),

        # Keep both names:
        # evidence_type: compatible with older Step5/Step6 scripts
        # pair_evidence_type: faithful to Step3 naming
        "evidence_type": pair_evidence_type,
        "pair_evidence_type": pair_evidence_type,

        "probe_direction_relative_to_text": probe_direction,
        "is_inverse_of_text_relation": is_inverse_of_text_relation,

        # Geometry / source relation metadata
        "geometry_label": pair_target.get("geometry_label"),
        "source_relation_summary": source_relation_summary,

        # Character-span hints for Step 5 validation.
        # Step 5 still tokenizes the text and computes token spans itself;
        # these spans are only used for checks and debugging.
        "subject_all_char_spans": subject_all_char_spans,
        "object_all_char_spans": object_all_char_spans,
        "subject_mention_count": len(subject_all_char_spans),
        "object_mention_count": len(object_all_char_spans),

        # Paragraph diagnostics
        "paragraph_num_object_mentions": len(
            paragraph_record.get("object_mentions", [])
        ),
        "paragraph_num_pair_targets": len(
            paragraph_record.get("pair_targets", [])
        ),
        "paragraph_num_explicit_relations": len(
            paragraph_record.get("explicit_relations_in_text", [])
        ),

        "probe_pair": {
            "probe_subject_uid": subject_uid,
            "probe_object_uid": object_uid,
            "probe_relation_label": relation_label,
        },
        "evidence": {
            "evidence_type": pair_evidence_type,
            "probe_direction_relative_to_text": probe_direction,
            "is_inverse_of_text_relation": is_inverse_of_text_relation,
        },
    }

    return row


print("Pair-level Step4 sample builder loaded.")

Pair-level Step4 sample builder loaded.


In [ ]:
# Composition / implicit-transitive helper functions

def build_explicit_relation_structures(
    paragraph: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Build explicit relation graph from paragraph["explicit_relations_in_text"].
    """
    explicit_edges = []
    explicit_edge_set = set()
    explicit_unordered_pairs = set()

    for rel_idx, rel in enumerate(paragraph.get("explicit_relations_in_text", [])):
        subject_uid = rel.get("subject_uid")
        relation = rel.get("relation")
        object_uid = rel.get("object_uid")

        if not subject_uid or not relation or not object_uid:
            continue

        edge = {
            "relation_index": rel_idx,
            "subject_uid": subject_uid,
            "subject_id": rel.get("subject_id"),
            "subject_type": rel.get("subject_type"),

            "relation": relation,

            "object_uid": object_uid,
            "object_id": rel.get("object_id"),
            "object_type": rel.get("object_type"),

            "sentence": rel.get("sentence"),
            "template": rel.get("template"),
            "template_index": rel.get("template_index"),
            "generation_mode": rel.get("generation_mode"),

            "was_swapped_for_text": rel.get("was_swapped_for_text"),
            "source_relation": rel.get("source_relation"),
        }

        explicit_edges.append(edge)
        explicit_edge_set.add(
            make_relation_key(subject_uid, relation, object_uid)
        )
        explicit_unordered_pairs.add(
            make_unordered_pair(subject_uid, object_uid)
        )

    return {
        "explicit_edges": explicit_edges,
        "explicit_edge_set": explicit_edge_set,
        "explicit_unordered_pairs": explicit_unordered_pairs,
    }


def build_pair_target_map(
    paragraph: Dict[str, Any],
) -> Dict[Tuple[str, str], Dict[str, Any]]:
    """
    Map directed pair:
        (subject_uid, object_uid) -> pair_target
    """
    pair_map = {}

    for pt in paragraph.get("pair_targets", []):
        subject_uid = pt.get("subject_uid")
        object_uid = pt.get("object_uid")

        if subject_uid and object_uid:
            pair_map[(subject_uid, object_uid)] = pt

    return pair_map


def is_target_explicit_direct_or_inverse(
    target_subject_uid: str,
    target_relation: str,
    target_object_uid: str,
    explicit_edge_set: set,
) -> bool:
    """
    Check whether target A-R-C or inverse C-inv(R)-A appears explicitly.
    """
    direct_key = make_relation_key(
        target_subject_uid,
        target_relation,
        target_object_uid,
    )

    inverse_key = make_relation_key(
        target_object_uid,
        inverse_relation(target_relation),
        target_subject_uid,
    )

    return direct_key in explicit_edge_set or inverse_key in explicit_edge_set


def is_target_pair_explicit_any_relation(
    target_subject_uid: str,
    target_object_uid: str,
    explicit_unordered_pairs: set,
) -> bool:
    """
    Strong leakage check:
    if A and C appear as any explicit pair in text, exclude the target.
    """
    return make_unordered_pair(
        target_subject_uid,
        target_object_uid,
    ) in explicit_unordered_pairs


def get_explicit_edge(
    explicit_edge_lookup: Dict[Tuple[str, str, str], Dict[str, Any]],
    subject_uid: str,
    relation: str,
    object_uid: str,
) -> Optional[Dict[str, Any]]:
    return explicit_edge_lookup.get(
        make_relation_key(subject_uid, relation, object_uid)
    )


def compact_support_edge(edge: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "subject_uid": edge.get("subject_uid"),
        "subject_id": edge.get("subject_id"),
        "subject_type": edge.get("subject_type"),

        "relation": edge.get("relation"),

        "object_uid": edge.get("object_uid"),
        "object_id": edge.get("object_id"),
        "object_type": edge.get("object_type"),

        "sentence": edge.get("sentence"),
        "relation_index": edge.get("relation_index"),
        "source_relation": edge.get("source_relation"),
    }


def find_all_supporting_paths(
    target_subject_uid: str,
    target_relation: str,
    target_object_uid: str,
    explicit_edges: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """
    Find explicit two-hop support paths for:
        A --R--> C

    Allowed patterns:
        1. chain_same_direction:
           A --R--> B, B --R--> C

        2. shared_anchor_opposite_sides:
           A --R--> B, C --inverse(R)--> B

        3. anchor_between_reversed_surface_form:
           B --inverse(R)--> A, B --R--> C
    """
    A = target_subject_uid
    R = target_relation
    C = target_object_uid
    inv_R = inverse_relation(R)

    explicit_edge_lookup = {
        make_relation_key(
            edge["subject_uid"],
            edge["relation"],
            edge["object_uid"],
        ): edge
        for edge in explicit_edges
    }

    object_uids = sorted(
        {
            edge["subject_uid"]
            for edge in explicit_edges
        }
        |
        {
            edge["object_uid"]
            for edge in explicit_edges
        }
    )

    supports = []
    seen_support_keys = set()

    for B in object_uids:
        if B in {A, C}:
            continue

        # Rule 1:
        # A R B + B R C -> A R C
        if "chain_same_direction" in STEP4_COMPOSITION_ALLOWED_RULES:
            edge1 = get_explicit_edge(explicit_edge_lookup, A, R, B)
            edge2 = get_explicit_edge(explicit_edge_lookup, B, R, C)

            if edge1 is not None and edge2 is not None:
                support_key = (
                    "chain_same_direction",
                    A,
                    R,
                    B,
                    R,
                    C,
                )

                if support_key not in seen_support_keys:
                    supports.append({
                        "implicit_rule": "chain_same_direction",
                        "bridge_uid": B,
                        "premise_1": compact_support_edge(edge1),
                        "premise_2": compact_support_edge(edge2),
                    })
                    seen_support_keys.add(support_key)

        # A R B + C inv(R) B -> A R C
        if "shared_anchor_opposite_sides" in STEP4_COMPOSITION_ALLOWED_RULES:
            edge1 = get_explicit_edge(explicit_edge_lookup, A, R, B)
            edge2 = get_explicit_edge(explicit_edge_lookup, C, inv_R, B)

            if edge1 is not None and edge2 is not None:
                support_key = (
                    "shared_anchor_opposite_sides",
                    A,
                    R,
                    B,
                    C,
                    inv_R,
                    B,
                )

                if support_key not in seen_support_keys:
                    supports.append({
                        "implicit_rule": "shared_anchor_opposite_sides",
                        "bridge_uid": B,
                        "premise_1": compact_support_edge(edge1),
                        "premise_2": compact_support_edge(edge2),
                    })
                    seen_support_keys.add(support_key)

        # Rule 3:
        # B inv(R) A + B R C -> A R C
        if "anchor_between_reversed_surface_form" in STEP4_COMPOSITION_ALLOWED_RULES:
            edge1 = get_explicit_edge(explicit_edge_lookup, B, inv_R, A)
            edge2 = get_explicit_edge(explicit_edge_lookup, B, R, C)

            if edge1 is not None and edge2 is not None:
                support_key = (
                    "anchor_between_reversed_surface_form",
                    B,
                    inv_R,
                    A,
                    B,
                    R,
                    C,
                )

                if support_key not in seen_support_keys:
                    supports.append({
                        "implicit_rule": "anchor_between_reversed_surface_form",
                        "bridge_uid": B,
                        "premise_1": compact_support_edge(edge1),
                        "premise_2": compact_support_edge(edge2),
                    })
                    seen_support_keys.add(support_key)

    return supports


def choose_primary_support(
    supporting_paths: List[Dict[str, Any]]
) -> Optional[Dict[str, Any]]:
    """
    Choose one support path for compact display.

    All support paths are still saved in the full sample.
    """
    if not supporting_paths:
        return None

    rule_priority = {
        "chain_same_direction": 0,
        "shared_anchor_opposite_sides": 1,
        "anchor_between_reversed_surface_form": 2,
    }

    return sorted(
        supporting_paths,
        key=lambda x: (
            rule_priority.get(x.get("implicit_rule"), 99),
            x.get("bridge_uid", ""),
        ),
    )[0]


print("Composition helper functions loaded.")

Composition helper functions loaded.


In [ ]:
# Build composition / implicit-transitive sample

def build_composition_sample(
    paragraph_record: Dict[str, Any],
    target_pair: Dict[str, Any],
    supporting_paths: List[Dict[str, Any]],
) -> Dict[str, Any]:
    paragraph_id = paragraph_record["paragraph_id"]
    text = paragraph_record["text"]

    target_subject_uid = target_pair["subject_uid"]
    target_object_uid = target_pair["object_uid"]
    target_relation = target_pair["relation_label"]

    sample_id = make_composition_sample_id(
        paragraph_id=paragraph_id,
        target_subject_uid=target_subject_uid,
        target_relation=target_relation,
        target_object_uid=target_object_uid,
    )

    pair_group_id = make_composition_group_id(
        paragraph_id=paragraph_id,
        target_subject_uid=target_subject_uid,
        target_relation=target_relation,
        target_object_uid=target_object_uid,
    )

    primary_support = choose_primary_support(supporting_paths)

    bridge_uids = sorted(
        {
            path.get("bridge_uid")
            for path in supporting_paths
            if path.get("bridge_uid") is not None
        }
    )

    implicit_rules = sorted(
        {
            path.get("implicit_rule")
            for path in supporting_paths
            if path.get("implicit_rule") is not None
        }
    )

    subject_all_char_spans = find_all_spans(text, target_subject_uid)
    object_all_char_spans = find_all_spans(text, target_object_uid)

    row = {
        # Identity
        "sample_id": sample_id,
        "sample_type": "composition",
        "composition_type": IMPLICIT_TRANSITIVE,
        "pair_group_id": pair_group_id,

        "paragraph_id": paragraph_id,
        "scene": paragraph_record["scene"],
        "scene_type": paragraph_record.get("scene_type"),
        "chunk_id": paragraph_record["chunk_id"],
        "cluster_id": paragraph_record["cluster_id"],
        "generation_mode": paragraph_record.get("generation_mode"),
        "paragraph_index_within_cluster": paragraph_record.get(
            "paragraph_index_within_cluster"
        ),

        # Text input for Step5
        "text": text,

        # Target pair A-C
        # These names intentionally mirror pair-level samples so Step5 can
        # extract hidden(A) - hidden(C) without special casing too much.
        "subject_uid": target_subject_uid,
        "subject_id": target_pair.get("subject_id"),
        "subject_type": target_pair.get("subject_type"),

        "object_uid": target_object_uid,
        "object_id": target_pair.get("object_id"),
        "object_type": target_pair.get("object_type"),

        "target_subject_uid": target_subject_uid,
        "target_object_uid": target_object_uid,
        "target_relation": target_relation,

        # Label fields
        "relation": target_relation,
        "relation_label": target_relation,
        "has_relation_label": target_pair.get("has_relation_label", True),
        "label_is_none": target_relation == NONE_RELATION_LABEL,

        # Evidence fields
        "relation_source": target_pair.get("relation_source"),

        "evidence_type": IMPLICIT_TRANSITIVE,
        "pair_evidence_type": IMPLICIT_TRANSITIVE,
        "target_pair_evidence_type": target_pair.get("pair_evidence_type"),

        "probe_direction_relative_to_text": "implicit_transitive",
        "is_inverse_of_text_relation": False,

        # Target metadata from Step3 pair_targets
        "geometry_label": target_pair.get("geometry_label"),
        "source_relation_summary": target_pair.get("source_relation_summary"),

        # Composition evidence
        "num_supporting_paths": len(supporting_paths),
        "supporting_paths": supporting_paths,
        "primary_support": primary_support,
        "bridge_uids": bridge_uids,
        "num_bridge_uids": len(bridge_uids),
        "implicit_rules": implicit_rules,

        # Leakage flags
        # These should be false by construction.
        "target_direct_or_inverse_explicit_leakage": False,
        "target_any_explicit_pair_leakage": False,

        # Span hints for Step5 validation
        "subject_all_char_spans": subject_all_char_spans,
        "object_all_char_spans": object_all_char_spans,
        "subject_mention_count": len(subject_all_char_spans),
        "object_mention_count": len(object_all_char_spans),

        # Paragraph diagnostics
        "paragraph_num_object_mentions": len(
            paragraph_record.get("object_mentions", [])
        ),
        "paragraph_num_pair_targets": len(
            paragraph_record.get("pair_targets", [])
        ),
        "paragraph_num_explicit_relations": len(
            paragraph_record.get("explicit_relations_in_text", [])
        ),

        "probe_pair": {
            "probe_subject_uid": target_subject_uid,
            "probe_object_uid": target_object_uid,
            "probe_relation_label": target_relation,
            "is_explicit_in_text": False,
            "is_implicit_transitive": True,
        },
        "evidence": {
            "evidence_type": IMPLICIT_TRANSITIVE,
            "probe_direction_relative_to_text": "implicit_transitive",
            "is_inverse_of_text_relation": False,
            "supporting_relations": supporting_paths,
            "primary_support": primary_support,
            "num_supporting_paths": len(supporting_paths),
            "bridge_uids": bridge_uids,
            "implicit_rules": implicit_rules,
        },
    }

    return row


print("Composition sample builder loaded.")

Composition sample builder loaded.


In [ ]:
# Build Step4 pair samples and composition samples from Step3 scene JSON files

all_step4_samples = []
all_composition_samples = []

scene_summaries = []
paragraph_summaries = []

composition_skip_counts = Counter()

for filename in step3_scene_files:
    path = os.path.join(STEP4_INPUT_DIR, filename)
    scene_data = load_json(path)

    validate_step3_scene_schema(scene_data, path)

    scene = scene_data["scene"]
    scene_type = scene_data.get("scene_type")

    scene_pair_samples = []
    scene_composition_samples = []

    num_paragraphs = 0
    num_pair_targets = 0

    for paragraph in scene_data["paragraphs"]:
        validate_step3_paragraph_schema(paragraph)

        num_paragraphs += 1

        pair_targets = paragraph.get("pair_targets", [])
        num_pair_targets += len(pair_targets)

        paragraph_pair_sample_count = 0
        paragraph_composition_sample_count = 0

        # Build ordinary pair-level samples
        for pair_index, pt in enumerate(pair_targets):
            evidence_type = pt.get("pair_evidence_type")
            relation_label = pt.get("relation_label")

            if evidence_type not in STEP4_KEEP_EVIDENCE_TYPES:
                continue

            if not STEP4_INCLUDE_NONE_LABEL and relation_label == NONE_RELATION_LABEL:
                continue

            row = build_step4_sample(
                paragraph_record=paragraph,
                pair_target=pt,
                pair_index=pair_index,
            )

            scene_pair_samples.append(row)
            all_step4_samples.append(row)
            paragraph_pair_sample_count += 1

        # Build composition / implicit-transitive samples
        explicit_structures = build_explicit_relation_structures(paragraph)
        explicit_edges = explicit_structures["explicit_edges"]
        explicit_edge_set = explicit_structures["explicit_edge_set"]
        explicit_unordered_pairs = explicit_structures["explicit_unordered_pairs"]

        pair_target_map = build_pair_target_map(paragraph)

        for target_pair in pair_targets:
            target_evidence_type = target_pair.get("pair_evidence_type")
            target_relation = target_pair.get("relation_label")

            A = target_pair.get("subject_uid")
            C = target_pair.get("object_uid")

            if not A or not C or not target_relation:
                composition_skip_counts["missing_target_fields"] += 1
                continue

            if target_relation not in STEP4_COMPOSITION_TARGET_RELATIONS:
                composition_skip_counts["target_relation_not_allowed"] += 1
                continue

            if STEP4_COMPOSITION_REQUIRE_TARGET_IMPLICIT_LABELED:
                if target_evidence_type != IMPLICIT_LABELED:
                    composition_skip_counts["target_not_implicit_labeled"] += 1
                    continue

            if target_relation == NONE_RELATION_LABEL:
                composition_skip_counts["target_none_relation"] += 1
                continue

            # Exclude A-C or C-A explicit leakage.
            if STEP4_COMPOSITION_EXCLUDE_DIRECT_OR_INVERSE_EXPLICIT_TARGET:
                if is_target_explicit_direct_or_inverse(
                    target_subject_uid=A,
                    target_relation=target_relation,
                    target_object_uid=C,
                    explicit_edge_set=explicit_edge_set,
                ):
                    composition_skip_counts["target_direct_or_inverse_explicit_leakage"] += 1
                    continue

            # Stronger leakage control:
            # exclude if A and C appear as any explicit pair.
            if STEP4_COMPOSITION_EXCLUDE_ANY_EXPLICIT_TARGET_PAIR:
                if is_target_pair_explicit_any_relation(
                    target_subject_uid=A,
                    target_object_uid=C,
                    explicit_unordered_pairs=explicit_unordered_pairs,
                ):
                    composition_skip_counts["target_any_explicit_pair_leakage"] += 1
                    continue

            supporting_paths = find_all_supporting_paths(
                target_subject_uid=A,
                target_relation=target_relation,
                target_object_uid=C,
                explicit_edges=explicit_edges,
            )

            if not supporting_paths:
                composition_skip_counts["no_supporting_path"] += 1
                continue

            if STEP4_COMPOSITION_REQUIRE_UNIQUE_SUPPORT_PATH and len(supporting_paths) != 1:
                composition_skip_counts["non_unique_supporting_path"] += 1
                continue

            if not STEP4_COMPOSITION_KEEP_ALL_SUPPORT_PATHS:
                primary_support = choose_primary_support(supporting_paths)
                supporting_paths = [primary_support] if primary_support is not None else []

            comp_row = build_composition_sample(
                paragraph_record=paragraph,
                target_pair=target_pair,
                supporting_paths=supporting_paths,
            )

            scene_composition_samples.append(comp_row)
            all_composition_samples.append(comp_row)
            paragraph_composition_sample_count += 1

        paragraph_summaries.append({
            "paragraph_id": paragraph["paragraph_id"],
            "scene": paragraph["scene"],
            "scene_type": paragraph.get("scene_type"),
            "chunk_id": paragraph["chunk_id"],
            "cluster_id": paragraph["cluster_id"],
            "num_pair_targets": len(pair_targets),
            "num_step4_pair_samples": paragraph_pair_sample_count,
            "num_step4_composition_samples": paragraph_composition_sample_count,
        })

    scene_summaries.append({
        "scene": scene,
        "scene_type": scene_type,
        "filename": filename,
        "num_paragraphs": num_paragraphs,
        "num_pair_targets": num_pair_targets,
        "num_step4_pair_samples": len(scene_pair_samples),
        "num_step4_composition_samples": len(scene_composition_samples),
    })

    print(
        f"[Loaded] {filename} | "
        f"scene={scene} | "
        f"scene_type={scene_type} | "
        f"paragraphs={num_paragraphs} | "
        f"pair_targets={num_pair_targets} | "
        f"pair_samples={len(scene_pair_samples)} | "
        f"composition_samples={len(scene_composition_samples)}"
    )

print("\nTotal ordinary Step4 pair samples:", len(all_step4_samples))
print("Total composition samples:", len(all_composition_samples))
print("Total scenes:", len(scene_summaries))
print("Total paragraphs:", len(paragraph_summaries))

print("\nPair evidence type counts:")
print(json.dumps(
    dict(Counter(row["evidence_type"] for row in all_step4_samples)),
    indent=2,
    ensure_ascii=False,
))

print("\nPair relation counts:")
print(json.dumps(
    dict(Counter(row["relation"] for row in all_step4_samples)),
    indent=2,
    ensure_ascii=False,
))

print("\nComposition relation counts:")
print(json.dumps(
    dict(Counter(row["relation"] for row in all_composition_samples)),
    indent=2,
    ensure_ascii=False,
))

print("\nComposition skip counts:")
print(json.dumps(
    dict(composition_skip_counts),
    indent=2,
    ensure_ascii=False,
))

[Loaded] FloorPlan1_step3_text_diverse.json | scene=FloorPlan1 | scene_type=kitchen | paragraphs=24 | pair_targets=2666 | pair_samples=2666 | composition_samples=29
[Loaded] FloorPlan2_step3_text_diverse.json | scene=FloorPlan2 | scene_type=kitchen | paragraphs=20 | pair_targets=1856 | pair_samples=1856 | composition_samples=18
[Loaded] FloorPlan3_step3_text_diverse.json | scene=FloorPlan3 | scene_type=kitchen | paragraphs=24 | pair_targets=1432 | pair_samples=1432 | composition_samples=5
[Loaded] FloorPlan4_step3_text_diverse.json | scene=FloorPlan4 | scene_type=kitchen | paragraphs=20 | pair_targets=2304 | pair_samples=2304 | composition_samples=42
[Loaded] FloorPlan5_step3_text_diverse.json | scene=FloorPlan5 | scene_type=kitchen | paragraphs=24 | pair_targets=2520 | pair_samples=2520 | composition_samples=38
[Loaded] FloorPlan6_step3_text_diverse.json | scene=FloorPlan6 | scene_type=kitchen | paragraphs=24 | pair_targets=2210 | pair_samples=2210 | composition_samples=17
[Loaded] Fl

In [ ]:
def group_samples_by_scene(samples: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    grouped = defaultdict(list)

    for row in samples:
        grouped[row["scene"]].append(row)

    return dict(grouped)


scene_groups = group_samples_by_scene(all_step4_samples)

# Main Step4 pair-level output for Step5.
write_jsonl(STEP4_OUTPUT_FILE, all_step4_samples)

# Optional per-scene outputs for debugging.
per_scene_files = {}

if STEP4_WRITE_PER_SCENE_FILES:
    for scene, rows in sorted(scene_groups.items(), key=lambda x: natural_sort_key(x[0])):
        scene_path = os.path.join(
            STEP4_OUTPUT_DIR,
            f"{scene}_step4_probe_samples_{RUN_MODE}.jsonl",
        )

        write_jsonl(scene_path, rows)
        per_scene_files[scene] = scene_path

manifest = {
    "step": "step4",
    "sample_family": "pair",
    "input_source": STEP4_INPUT_SOURCE,
    "run_mode": RUN_MODE,

    "step4_input_dir": STEP4_INPUT_DIR,
    "step4_scene_json_suffix": STEP4_SCENE_JSON_SUFFIX,
    "step4_output_file": STEP4_OUTPUT_FILE,
    "step4_output_dir": STEP4_OUTPUT_DIR,

    "num_step3_scene_json_files": len(step3_scene_files),
    "num_available_scenes": len(available_scenes),
    "num_configured_scenes": len(SCENES),
    "num_missing_scenes": len(missing_scenes),
    "missing_scenes": missing_scenes,
    "extra_scenes": extra_scenes,

    "num_paragraphs": len(paragraph_summaries),
    "num_probe_samples": len(all_step4_samples),
    "num_composition_samples": len(all_composition_samples),
    "num_scenes_with_samples": len(scene_groups),
    "scenes_with_samples": sorted(scene_groups.keys(), key=natural_sort_key),

    "evidence_type_counts": dict(
        Counter(row["evidence_type"] for row in all_step4_samples)
    ),
    "relation_counts": dict(
        Counter(row["relation"] for row in all_step4_samples)
    ),
    "label_is_none_counts": dict(
        Counter(str(row["label_is_none"]) for row in all_step4_samples)
    ),

    "keep_evidence_types": sorted(STEP4_KEEP_EVIDENCE_TYPES),
    "include_none_label": STEP4_INCLUDE_NONE_LABEL,

    "write_per_scene_files": STEP4_WRITE_PER_SCENE_FILES,
    "per_scene_files": per_scene_files,

    "scene_summaries": scene_summaries,
}

write_json(STEP4_MANIFEST_FILE, manifest)

print("Wrote ordinary Step4 pair-level outputs:")
print("STEP4_OUTPUT_FILE:", STEP4_OUTPUT_FILE)
print("STEP4_MANIFEST_FILE:", STEP4_MANIFEST_FILE)

if STEP4_WRITE_PER_SCENE_FILES:
    print("Per-scene pair files written:", len(per_scene_files))

print("\nPair manifest summary:")
print(json.dumps(
    {
        "num_probe_samples": manifest["num_probe_samples"],
        "num_paragraphs": manifest["num_paragraphs"],
        "num_scenes_with_samples": manifest["num_scenes_with_samples"],
        "evidence_type_counts": manifest["evidence_type_counts"],
        "relation_counts": manifest["relation_counts"],
    },
    indent=2,
    ensure_ascii=False,
))

Wrote ordinary Step4 pair-level outputs:
STEP4_OUTPUT_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_probe_samples_diverse_all.jsonl
STEP4_MANIFEST_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_manifest_diverse.json
Per-scene pair files written: 118

Pair manifest summary:
{
  "num_probe_samples": 153158,
  "num_paragraphs": 1932,
  "num_scenes_with_samples": 118,
  "evidence_type_counts": {
    "implicit_none": 58928,
    "implicit_labeled": 46414,
    "explicit_direct": 24252,
    "explicit_inverse_or_same_sentence_pair": 23564
  },
  "relation_counts": {
    "none": 58928,
    "right_of": 22530,
    "above": 6276,
    "contains": 1076,
    "near": 16774,
    "on": 8846,
    "supports": 8840,
    "below": 6276,
    "in": 1082,
    "left_of": 22530
  }
}


In [ ]:
composition_scene_groups = group_samples_by_scene(all_composition_samples)

# Main composition output for Step5 composition mode.
write_jsonl(STEP4_COMPOSITION_OUTPUT_FILE, all_composition_samples)

composition_per_scene_files = {}

if STEP4_WRITE_COMPOSITION_PER_SCENE_FILES:
    for scene, rows in sorted(composition_scene_groups.items(), key=lambda x: natural_sort_key(x[0])):
        scene_path = os.path.join(
            STEP4_OUTPUT_DIR,
            f"{scene}_step4_composition_samples_{RUN_MODE}.jsonl",
        )

        write_jsonl(scene_path, rows)
        composition_per_scene_files[scene] = scene_path

composition_manifest = {
    "step": "step4",
    "sample_family": "composition",
    "composition_type": IMPLICIT_TRANSITIVE,
    "input_source": STEP4_INPUT_SOURCE,
    "run_mode": RUN_MODE,

    "step4_input_dir": STEP4_INPUT_DIR,
    "step4_scene_json_suffix": STEP4_SCENE_JSON_SUFFIX,
    "step4_composition_output_file": STEP4_COMPOSITION_OUTPUT_FILE,
    "step4_output_dir": STEP4_OUTPUT_DIR,

    "num_step3_scene_json_files": len(step3_scene_files),
    "num_available_scenes": len(available_scenes),
    "num_configured_scenes": len(SCENES),
    "num_missing_scenes": len(missing_scenes),
    "missing_scenes": missing_scenes,
    "extra_scenes": extra_scenes,

    "num_paragraphs": len(paragraph_summaries),
    "num_composition_samples": len(all_composition_samples),
    "num_scenes_with_composition_samples": len(composition_scene_groups),
    "scenes_with_composition_samples": sorted(
        composition_scene_groups.keys(),
        key=natural_sort_key,
    ),

    "target_relation_counts": dict(
        Counter(row["relation"] for row in all_composition_samples)
    ),
    "num_supporting_paths_counts": dict(
        Counter(str(row["num_supporting_paths"]) for row in all_composition_samples)
    ),
    "implicit_rule_counts": dict(
        Counter(
            rule
            for row in all_composition_samples
            for rule in row.get("implicit_rules", [])
        )
    ),

    "composition_skip_counts": dict(composition_skip_counts),

    "target_relations": sorted(STEP4_COMPOSITION_TARGET_RELATIONS),
    "allowed_rules": sorted(STEP4_COMPOSITION_ALLOWED_RULES),
    "require_target_implicit_labeled": STEP4_COMPOSITION_REQUIRE_TARGET_IMPLICIT_LABELED,
    "exclude_direct_or_inverse_explicit_target": STEP4_COMPOSITION_EXCLUDE_DIRECT_OR_INVERSE_EXPLICIT_TARGET,
    "exclude_any_explicit_target_pair": STEP4_COMPOSITION_EXCLUDE_ANY_EXPLICIT_TARGET_PAIR,
    "require_unique_support_path": STEP4_COMPOSITION_REQUIRE_UNIQUE_SUPPORT_PATH,
    "keep_all_support_paths": STEP4_COMPOSITION_KEEP_ALL_SUPPORT_PATHS,

    "write_composition_per_scene_files": STEP4_WRITE_COMPOSITION_PER_SCENE_FILES,
    "composition_per_scene_files": composition_per_scene_files,

    "scene_summaries": scene_summaries,
}

write_json(STEP4_COMPOSITION_MANIFEST_FILE, composition_manifest)

print("Wrote Step4 composition outputs:")
print("STEP4_COMPOSITION_OUTPUT_FILE:", STEP4_COMPOSITION_OUTPUT_FILE)
print("STEP4_COMPOSITION_MANIFEST_FILE:", STEP4_COMPOSITION_MANIFEST_FILE)

if STEP4_WRITE_COMPOSITION_PER_SCENE_FILES:
    print("Per-scene composition files written:", len(composition_per_scene_files))

print("\nComposition manifest summary:")
print(json.dumps(
    {
        "num_composition_samples": composition_manifest["num_composition_samples"],
        "num_scenes_with_composition_samples": composition_manifest["num_scenes_with_composition_samples"],
        "target_relation_counts": composition_manifest["target_relation_counts"],
        "num_supporting_paths_counts": composition_manifest["num_supporting_paths_counts"],
        "implicit_rule_counts": composition_manifest["implicit_rule_counts"],
        "composition_skip_counts": composition_manifest["composition_skip_counts"],
    },
    indent=2,
    ensure_ascii=False,
))

Wrote Step4 composition outputs:
STEP4_COMPOSITION_OUTPUT_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_composition_samples_diverse_all.jsonl
STEP4_COMPOSITION_MANIFEST_FILE: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_composition_manifest_diverse.json
Per-scene composition files written: 94

Composition manifest summary:
{
  "num_composition_samples": 1577,
  "num_scenes_with_composition_samples": 94,
  "target_relation_counts": {
    "right_of": 754,
    "left_of": 772,
    "below": 23,
    "above": 28
  },
  "num_supporting_paths_counts": {
    "1": 1492,
    "2": 82,
    "3": 3
  },
  "implicit_rule_counts": {
    "chain_same_direction": 588,
    "anchor_between_reversed_surface_form": 570,
    "shared_anchor_opposite_sides": 476
  },
  "composition_skip_counts": {
    "target_relation_not_allowed": 95546,
    "no_supporting_path": 28191,
    "target_not_implicit_la

In [ ]:
def make_preview_row(row: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "sample_id": row["sample_id"],
        "sample_type": row["sample_type"],
        "pair_group_id": row["pair_group_id"],

        "scene": row["scene"],
        "scene_type": row.get("scene_type"),
        "paragraph_id": row["paragraph_id"],
        "chunk_id": row["chunk_id"],
        "cluster_id": row["cluster_id"],

        "relation": row["relation"],
        "evidence_type": row["evidence_type"],
        "pair_evidence_type": row["pair_evidence_type"],
        "probe_direction_relative_to_text": row["probe_direction_relative_to_text"],
        "is_inverse_of_text_relation": row["is_inverse_of_text_relation"],

        "subject_uid": row["subject_uid"],
        "object_uid": row["object_uid"],

        "subject_type": row.get("subject_type"),
        "object_type": row.get("object_type"),

        "subject_mention_count": row["subject_mention_count"],
        "object_mention_count": row["object_mention_count"],

        "has_relation_label": row["has_relation_label"],
        "label_is_none": row["label_is_none"],

        "text": row["text"],
    }


preview_rows = [
    make_preview_row(row)
    for row in all_step4_samples
]

df_preview = pd.DataFrame(preview_rows)

df_preview.to_csv(
    STEP4_PREVIEW_CSV_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Pair preview CSV:", STEP4_PREVIEW_CSV_FILE)
print("Pair preview rows:", len(df_preview))

display(df_preview.head(20))

Pair preview CSV: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_probe_samples_diverse_preview.csv
Pair preview rows: 153158


,sample_id,sample_type,pair_group_id,scene,scene_type,paragraph_id,chunk_id,cluster_id,relation,evidence_type,...,is_inverse_of_text_relation,subject_uid,object_uid,subject_type,object_type,subject_mention_count,object_mention_count,has_relation_label,label_is_none,text
0,s4_ca13d8e0f1b83f7621536ab0,pair,pg_a404b541b24669c946f2,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,none,implicit_none,...,False,cabinet_1,cabinet_2,Cabinet,Cabinet,2,1,False,True,"In the described kitchen area, coffee_machine_..."
1,s4_3e9f5c1b26625b60096f61b8,pair,pg_f407805e2aa8345b3230,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,none,implicit_none,...,False,cabinet_1,coffee_machine_1,Cabinet,CoffeeMachine,2,1,False,True,"In the described kitchen area, coffee_machine_..."
2,s4_09780f0bc6977d2c6f0c35e8,pair,pg_053d9eea14b51dc79f86,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,none,implicit_none,...,False,cabinet_1,counter_top_1,Cabinet,CounterTop,2,9,False,True,"In the described kitchen area, coffee_machine_..."
3,s4_12c891b3d79ec182f9ef5181,pair,pg_a73c02e93c6355b983a2,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,none,implicit_none,...,False,cabinet_1,dish_sponge_1,Cabinet,DishSponge,2,4,False,True,"In the described kitchen area, coffee_machine_..."
4,s4_91ae87687de31027bb9c9d29,pair,pg_6c8f3defbcf376f0f503,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,none,implicit_none,...,False,cabinet_1,house_plant_1,Cabinet,HousePlant,2,2,False,True,"In the described kitchen area, coffee_machine_..."
5,s4_6b4716ee3f10c86be1a8fe6c,pair,pg_c3bcbb4dd1f9071fceb8,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,none,implicit_none,...,False,cabinet_1,lettuce_1,Cabinet,Lettuce,2,2,False,True,"In the described kitchen area, coffee_machine_..."
6,s4_9eafe221ca8daf0c7b03af68,pair,pg_27742753863d5df73d6d,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,none,implicit_none,...,False,cabinet_1,mug_1,Cabinet,Mug,2,1,False,True,"In the described kitchen area, coffee_machine_..."
7,s4_ff1620e52f6013713daba114,pair,pg_658c762049ed0097af0c,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,none,implicit_none,...,False,cabinet_1,paper_towel_roll_1,Cabinet,PaperTowelRoll,2,2,False,True,"In the described kitchen area, coffee_machine_..."
8,s4_cff2dda4ff71187acd08c9f8,pair,pg_2b353406fac5277380b7,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,right_of,implicit_labeled,...,False,cabinet_1,potato_1,Cabinet,Potato,2,2,True,False,"In the described kitchen area, coffee_machine_..."
9,s4_e313e62718eaedc3182f6f70,pair,pg_e41d676be99a9d275cdf,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step...,FloorPlan1_chunk_0_0,FloorPlan1_chunk_0_0_cluster_0,above,explicit_direct,...,False,cabinet_1,pot_1,Cabinet,Pot,2,4,True,False,"In the described kitchen area, coffee_machine_..."


In [ ]:
def summarize_primary_support(row: Dict[str, Any]) -> str:
    primary = row.get("primary_support")

    if not primary:
        return ""

    p1 = primary.get("premise_1", {})
    p2 = primary.get("premise_2", {})

    s1 = f"{p1.get('subject_uid')} {p1.get('relation')} {p1.get('object_uid')}"
    s2 = f"{p2.get('subject_uid')} {p2.get('relation')} {p2.get('object_uid')}"

    return f"{s1} ; {s2}"


def make_composition_preview_row(row: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "sample_id": row["sample_id"],
        "sample_type": row["sample_type"],
        "composition_type": row["composition_type"],
        "pair_group_id": row["pair_group_id"],

        "scene": row["scene"],
        "scene_type": row.get("scene_type"),
        "paragraph_id": row["paragraph_id"],
        "chunk_id": row["chunk_id"],
        "cluster_id": row["cluster_id"],

        "target_subject_uid": row["target_subject_uid"],
        "target_relation": row["target_relation"],
        "target_object_uid": row["target_object_uid"],

        "relation": row["relation"],
        "evidence_type": row["evidence_type"],
        "target_pair_evidence_type": row["target_pair_evidence_type"],

        "num_supporting_paths": row["num_supporting_paths"],
        "bridge_uids": "|".join(row.get("bridge_uids", [])),
        "implicit_rules": "|".join(row.get("implicit_rules", [])),
        "primary_support": summarize_primary_support(row),

        "subject_mention_count": row["subject_mention_count"],
        "object_mention_count": row["object_mention_count"],

        "text": row["text"],
    }


composition_preview_rows = [
    make_composition_preview_row(row)
    for row in all_composition_samples
]

df_composition_preview = pd.DataFrame(composition_preview_rows)

df_composition_preview.to_csv(
    STEP4_COMPOSITION_PREVIEW_CSV_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Composition preview CSV:", STEP4_COMPOSITION_PREVIEW_CSV_FILE)
print("Composition preview rows:", len(df_composition_preview))

display(df_composition_preview.head(20))

Composition preview CSV: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe/step4_composition_samples_diverse_preview.csv
Composition preview rows: 1577


,sample_id,sample_type,composition_type,pair_group_id,scene,scene_type,paragraph_id,chunk_id,cluster_id,target_subject_uid,...,relation,evidence_type,target_pair_evidence_type,num_supporting_paths,bridge_uids,implicit_rules,primary_support,subject_mention_count,object_mention_count,text
0,s4t_5c17f6772b2fb470739435c9,composition,implicit_transitive,cpg_fa778c3dbb17661318660275,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_0_1_cluster_0_step...,FloorPlan1_chunk_0_1,FloorPlan1_chunk_0_1_cluster_0,pot_1,...,right_of,implicit_transitive,implicit_labeled,1,potato_1,chain_same_direction,pot_1 right_of potato_1 ; potato_1 right_of so...,3,2,"In the described kitchen area, On top of count..."
1,s4t_310e92871f9b308bf1e66dfc,composition,implicit_transitive,cpg_7094b9566d6c416226924b2f,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_step...,FloorPlan1_chunk_1_0,FloorPlan1_chunk_1_0_cluster_0,kettle_1,...,right_of,implicit_transitive,implicit_labeled,1,pan_1,anchor_between_reversed_surface_form,pan_1 left_of kettle_1 ; pan_1 right_of salt_s...,2,3,"In this part of the kitchen, Directly below ca..."
2,s4t_888bf688848a5c7a535e3bf3,composition,implicit_transitive,cpg_158e44b45fa5705e54f62198,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_step...,FloorPlan1_chunk_1_0,FloorPlan1_chunk_1_0_cluster_0,kettle_1,...,right_of,implicit_transitive,implicit_labeled,1,pan_1,anchor_between_reversed_surface_form,pan_1 left_of kettle_1 ; pan_1 right_of stove_...,2,3,"In this part of the kitchen, Directly below ca..."
3,s4t_ba9a8e42b90ea5e54419be83,composition,implicit_transitive,cpg_7ef0fd6733cf77cf0d637467,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_step...,FloorPlan1_chunk_1_0,FloorPlan1_chunk_1_0_cluster_0,plate_1,...,right_of,implicit_transitive,implicit_labeled,1,spatula_1,shared_anchor_opposite_sides,plate_1 right_of spatula_1 ; stove_burner_1 le...,3,1,"In this part of the kitchen, Directly below ca..."
4,s4t_e8f37ac7b30263a913edd84d,composition,implicit_transitive,cpg_f2e6433d26c17bb8161b0ac0,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_step...,FloorPlan1_chunk_1_0,FloorPlan1_chunk_1_0_cluster_0,plate_1,...,right_of,implicit_transitive,implicit_labeled,1,spatula_1,shared_anchor_opposite_sides,plate_1 right_of spatula_1 ; stove_burner_2 le...,3,2,"In this part of the kitchen, Directly below ca..."
5,s4t_af5236edbfbf11ae75ccd267,composition,implicit_transitive,cpg_225f3873fd5d8bb53eada4c9,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_step...,FloorPlan1_chunk_1_0,FloorPlan1_chunk_1_0_cluster_0,salt_shaker_1,...,left_of,implicit_transitive,implicit_labeled,1,pan_1,anchor_between_reversed_surface_form,pan_1 right_of salt_shaker_1 ; pan_1 left_of k...,3,2,"In this part of the kitchen, Directly below ca..."
6,s4t_222753014931dd5ac28e99fd,composition,implicit_transitive,cpg_6ffe56aa3102ddd58e87177d,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_step...,FloorPlan1_chunk_1_0,FloorPlan1_chunk_1_0_cluster_0,stove_burner_1,...,left_of,implicit_transitive,implicit_labeled,1,spatula_1,shared_anchor_opposite_sides,stove_burner_1 left_of spatula_1 ; plate_1 rig...,1,3,"In this part of the kitchen, Directly below ca..."
7,s4t_02c1952e6fdca949670b6274,composition,implicit_transitive,cpg_f83c09909a19892a1c8190a7,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_step...,FloorPlan1_chunk_1_0,FloorPlan1_chunk_1_0_cluster_0,stove_burner_2,...,left_of,implicit_transitive,implicit_labeled,1,spatula_1,shared_anchor_opposite_sides,stove_burner_2 left_of spatula_1 ; plate_1 rig...,2,3,"In this part of the kitchen, Directly below ca..."
8,s4t_a84ee0e67b77e32da7827e70,composition,implicit_transitive,cpg_99370c7601619a52291d0b04,FloorPlan1,kitchen,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_step...,FloorPlan1_chunk_1_0,FloorPlan1_chunk_1_0_cluster_0,stove_knob_1,...,left_of,implicit_transitive,implicit_labeled,1,pan_1,anchor_between_reversed_surface_form,pan_1 right_of stove_knob_1 

In [ ]:
required_pair_step5_fields = {
    "sample_id",
    "sample_type",
    "pair_group_id",
    "paragraph_id",
    "scene",
    "scene_type",
    "chunk_id",
    "cluster_id",
    "generation_mode",

    "text",

    "subject_uid",
    "subject_id",
    "subject_type",
    "object_uid",
    "object_id",
    "object_type",

    "relation",
    "relation_label",
    "has_relation_label",
    "relation_source",
    "evidence_type",
    "pair_evidence_type",

    "probe_direction_relative_to_text",
    "is_inverse_of_text_relation",

    "geometry_label",
    "source_relation_summary",

    "subject_all_char_spans",
    "object_all_char_spans",
    "subject_mention_count",
    "object_mention_count",
}

pair_missing_field_counter = Counter()
pair_bad_relation_mismatch = 0
pair_missing_subject_mentions = 0
pair_missing_object_mentions = 0

for row in all_step4_samples:
    for field in required_pair_step5_fields:
        if field not in row:
            pair_missing_field_counter[field] += 1

    if row.get("relation") != row.get("relation_label"):
        pair_bad_relation_mismatch += 1

    if row.get("subject_mention_count", 0) == 0:
        pair_missing_subject_mentions += 1

    if row.get("object_mention_count", 0) == 0:
        pair_missing_object_mentions += 1

print("Pair missing field counts:", dict(pair_missing_field_counter))
print("Pair relation != relation_label:", pair_bad_relation_mismatch)
print("Pair rows with no subject mention in text:", pair_missing_subject_mentions)
print("Pair rows with no object mention in text:", pair_missing_object_mentions)

if pair_missing_field_counter:
    raise ValueError(
        f"Some pair samples are missing required fields: {dict(pair_missing_field_counter)}"
    )

if pair_bad_relation_mismatch > 0:
    raise ValueError(
        f"Found {pair_bad_relation_mismatch} pair rows where relation != relation_label."
    )

# Composition checks

required_composition_fields = required_pair_step5_fields | {
    "composition_type",
    "target_subject_uid",
    "target_object_uid",
    "target_relation",
    "target_pair_evidence_type",
    "num_supporting_paths",
    "supporting_paths",
    "primary_support",
    "bridge_uids",
    "implicit_rules",
    "target_direct_or_inverse_explicit_leakage",
    "target_any_explicit_pair_leakage",
}

composition_missing_field_counter = Counter()
composition_bad_relation_mismatch = 0
composition_missing_subject_mentions = 0
composition_missing_object_mentions = 0
composition_bad_evidence_type = 0
composition_bad_target_evidence_type = 0
composition_no_supporting_paths = 0
composition_leakage_flags = 0

for row in all_composition_samples:
    for field in required_composition_fields:
        if field not in row:
            composition_missing_field_counter[field] += 1

    if row.get("relation") != row.get("relation_label"):
        composition_bad_relation_mismatch += 1

    if row.get("subject_mention_count", 0) == 0:
        composition_missing_subject_mentions += 1

    if row.get("object_mention_count", 0) == 0:
        composition_missing_object_mentions += 1

    if row.get("evidence_type") != IMPLICIT_TRANSITIVE:
        composition_bad_evidence_type += 1

    if row.get("target_pair_evidence_type") != IMPLICIT_LABELED:
        composition_bad_target_evidence_type += 1

    if row.get("num_supporting_paths", 0) < 1:
        composition_no_supporting_paths += 1

    if row.get("target_direct_or_inverse_explicit_leakage") or row.get("target_any_explicit_pair_leakage"):
        composition_leakage_flags += 1

print("\nComposition missing field counts:", dict(composition_missing_field_counter))
print("Composition relation != relation_label:", composition_bad_relation_mismatch)
print("Composition rows with no subject mention in text:", composition_missing_subject_mentions)
print("Composition rows with no object mention in text:", composition_missing_object_mentions)
print("Composition rows with wrong evidence_type:", composition_bad_evidence_type)
print("Composition rows with target_pair_evidence_type != implicit_labeled:", composition_bad_target_evidence_type)
print("Composition rows with no supporting paths:", composition_no_supporting_paths)
print("Composition rows with leakage flags:", composition_leakage_flags)

if composition_missing_field_counter:
    raise ValueError(
        f"Some composition samples are missing required fields: {dict(composition_missing_field_counter)}"
    )

if composition_bad_relation_mismatch > 0:
    raise ValueError(
        f"Found {composition_bad_relation_mismatch} composition rows where relation != relation_label."
    )

if composition_bad_evidence_type > 0:
    raise ValueError(
        f"Found {composition_bad_evidence_type} composition rows with wrong evidence_type."
    )

if composition_bad_target_evidence_type > 0:
    raise ValueError(
        f"Found {composition_bad_target_evidence_type} composition rows where target_pair_evidence_type != implicit_labeled."
    )

if composition_no_supporting_paths > 0:
    raise ValueError(
        f"Found {composition_no_supporting_paths} composition rows with no supporting paths."
    )

if composition_leakage_flags > 0:
    raise ValueError(
        f"Found {composition_leakage_flags} composition rows with leakage flags."
    )

if pair_missing_subject_mentions > 0 or pair_missing_object_mentions > 0:
    print(
        "[Warning] Some pair rows have missing subject/object mentions in text. "
        "Step5 should skip or inspect these rows."
    )

if composition_missing_subject_mentions > 0 or composition_missing_object_mentions > 0:
    print(
        "[Warning] Some composition rows have missing target subject/object mentions in text. "
        "Step5 should skip or inspect these rows."
    )

print("\nStep4 sanity checks completed.")

Pair missing field counts: {}
Pair relation != relation_label: 0
Pair rows with no subject mention in text: 0
Pair rows with no object mention in text: 0

Composition missing field counts: {}
Composition relation != relation_label: 0
Composition rows with no subject mention in text: 0
Composition rows with no object mention in text: 0
Composition rows with wrong evidence_type: 0
Composition rows with target_pair_evidence_type != implicit_labeled: 0
Composition rows with no supporting paths: 0
Composition rows with leakage flags: 0

Step4 sanity checks completed.


In [ ]:
for evidence_type in [
    EXPLICIT_DIRECT,
    EXPLICIT_INVERSE,
    IMPLICIT_LABELED,
    IMPLICIT_NONE,
]:
    rows = [
        row for row in all_step4_samples
        if row["evidence_type"] == evidence_type
    ]

    print("\n" + "=" * 80)
    print("Pair evidence type:", evidence_type)
    print("Num rows:", len(rows))

    for row in rows[:3]:
        print("-" * 80)
        print("sample_id:", row["sample_id"])
        print("pair_group_id:", row["pair_group_id"])
        print("scene:", row["scene"])
        print("paragraph_id:", row["paragraph_id"])
        print("relation:", row["relation"])
        print("subject:", row["subject_uid"])
        print("object:", row["object_uid"])
        print("subject_mention_count:", row["subject_mention_count"])
        print("object_mention_count:", row["object_mention_count"])
        print("text:", row["text"][:500])


print("\n" + "#" * 100)
print("Composition / implicit-transitive examples")
print("#" * 100)

for row in all_composition_samples[:10]:
    print("\n" + "=" * 80)
    print("sample_id:", row["sample_id"])
    print("scene:", row["scene"])
    print("paragraph_id:", row["paragraph_id"])
    print("target:", row["target_subject_uid"], row["target_relation"], row["target_object_uid"])
    print("num_supporting_paths:", row["num_supporting_paths"])
    print("bridge_uids:", row["bridge_uids"])
    print("implicit_rules:", row["implicit_rules"])

    primary = row.get("primary_support")
    if primary:
        print("primary premise 1:", primary["premise_1"]["subject_uid"], primary["premise_1"]["relation"], primary["premise_1"]["object_uid"])
        print("primary premise 2:", primary["premise_2"]["subject_uid"], primary["premise_2"]["relation"], primary["premise_2"]["object_uid"])

    print("text:", row["text"][:700])


Pair evidence type: explicit_direct
Num rows: 24252
--------------------------------------------------------------------------------
sample_id: s4_e313e62718eaedc3182f6f70
pair_group_id: pg_e41d676be99a9d275cdf
scene: FloorPlan1
paragraph_id: FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step3_diverse_0
relation: above
subject: cabinet_1
object: pot_1
subject_mention_count: 2
object_mention_count: 4
text: In the described kitchen area, coffee_machine_1 is on counter_top_1. On top of counter_top_1, there is mug_1. You can see pot_1 on counter_top_1. cabinet_1 is above pot_1. cabinet_2 has dish_sponge_1 inside it. cabinet_1 holds wine_bottle_1. counter_top_1 is holding up house_plant_1. On counter_top_1, there is lettuce_1. On counter_top_1, there is paper_towel_roll_1. counter_top_1 is positioned above dish_sponge_1. counter_top_1 sits below wine_bottle_1. On counter_top_1, there is potato_1. dish
--------------------------------------------------------------------------------
sample_id: s

In [ ]:
from google.colab import files

zip_base = f"/content/pilot_full_step4_outputs_{RUN_MODE}"

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    STEP4_OUTPUT_DIR,
)

print("Created:", zip_path)

files.download(zip_path)

Created: /content/pilot_full_step4_outputs_diverse.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>